# DA6401 Assignment 2 — Visual Perception Pipeline

Run every cell top to bottom in order. **T4 GPU required** — Runtime → Change runtime type → T4 GPU.

> If Colab resets mid-way, start from Cell 1 again. Cells 6 and 7 restore your checkpoints from Drive automatically.

## Cell 1 — GPU check

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change runtime!')

## Cell 2 — Clone repo and install deps

In [ ]:
import os

if not os.path.exists('/content/da6401_assignment_2'):
    !git clone https://github.com/Mohmad-Yaqoob/da6401_assignment_2.git
else:
    !cd /content/da6401_assignment_2 && git pull origin main

os.chdir('/content/da6401_assignment_2')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

!pip install -q wandb albumentations scikit-learn

## Cell 3 — Overwrite pets_dataset.py with clean version

This fixes all mask tensor bugs. Run this every session.

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')

code = r'''
import os, tarfile, urllib.request
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGES_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
ANNOTS_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

def download_and_extract(url, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    filename = url.split("/")[-1]
    filepath = os.path.join(dest_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} ...")
        urllib.request.urlretrieve(url, filepath)
        print("Done.")
    marker = filepath.replace(".tar.gz", "")
    if not os.path.exists(marker):
        print(f"Extracting {filename} ...")
        with tarfile.open(filepath, "r:gz") as tar:
            tar.extractall(dest_dir)
        print("Done.")

def prepare_dataset(root="./pets_data"):
    download_and_extract(IMAGES_URL, root)
    download_and_extract(ANNOTS_URL, root)
    return root

def parse_list_file(annotations_dir):
    entries = []
    with open(os.path.join(annotations_dir, "list.txt")) as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            parts = line.split()
            entries.append({"image_name": parts[0], "class_id": int(parts[1]) - 1})
    return entries

def parse_bbox_file(annotations_dir):
    import xml.etree.ElementTree as ET
    bbox_dir = os.path.join(annotations_dir, "xmls")
    bbox_map = {}
    if not os.path.exists(bbox_dir):
        return bbox_map
    for xf in os.listdir(bbox_dir):
        if not xf.endswith(".xml"):
            continue
        name = xf.replace(".xml", "")
        root = ET.parse(os.path.join(bbox_dir, xf)).getroot()
        obj  = root.find("object")
        if obj is None:
            continue
        b = obj.find("bndbox")
        bbox_map[name] = [float(b.find(t).text) for t in ["xmin","ymin","xmax","ymax"]]
    return bbox_map

def get_train_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.Affine(translate_percent=0.05, scale=(0.9,1.1), rotate=(-15,15), p=0.4),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["bbox_labels"], min_visibility=0.3))

def get_val_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["bbox_labels"], min_visibility=0.3))

class OxfordPetsDataset(Dataset):
    def __init__(self, root="./pets_data", split="train", img_size=224, val_fraction=0.15, seed=42):
        super().__init__()
        assert split in ("train","val","test")
        self.img_size = img_size
        images_dir      = os.path.join(root, "images")
        annotations_dir = os.path.join(root, "annotations")
        all_entries     = parse_list_file(annotations_dir)
        self.bbox_map   = parse_bbox_file(annotations_dir)
        tv_names, te_names = set(), set()
        for fname, s in [("trainval.txt", tv_names),("test.txt", te_names)]:
            with open(os.path.join(annotations_dir, fname)) as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("#") or not line: continue
                    s.add(line.split()[0])
        tv = [e for e in all_entries if e["image_name"] in tv_names]
        te = [e for e in all_entries if e["image_name"] in te_names]
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(tv)).tolist()
        cut = int(len(tv) * val_fraction)
        if split == "train":
            self.entries = [tv[i] for i in idx[cut:]]
            self.tfm     = get_train_transforms(img_size)
        elif split == "val":
            self.entries = [tv[i] for i in idx[:cut]]
            self.tfm     = get_val_transforms(img_size)
        else:
            self.entries = te
            self.tfm     = get_val_transforms(img_size)
        self.images_dir      = images_dir
        self.annotations_dir = annotations_dir
        print(f"[OxfordPets] {split}: {len(self.entries)} samples")

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        entry    = self.entries[idx]
        name     = entry["image_name"]
        class_id = entry["class_id"]
        img_np   = np.array(Image.open(os.path.join(self.images_dir, name+".jpg")).convert("RGB"))
        h, w     = img_np.shape[:2]

        has_bbox = name in self.bbox_map
        if has_bbox:
            xmin,ymin,xmax,ymax = self.bbox_map[name]
            xmin=max(0.,xmin); ymin=max(0.,ymin)
            xmax=min(float(w),xmax); ymax=min(float(h),ymax)
            bboxes=[[ xmin,ymin,xmax,ymax]]; blabels=[0]
        else:
            bboxes=[]; blabels=[]

        t = self.tfm(image=img_np, bboxes=bboxes, bbox_labels=blabels)
        image_tensor = t["image"]

        if has_bbox and len(t["bboxes"]) > 0:
            tx1,ty1,tx2,ty2 = t["bboxes"][0]
            S = self.img_size
            bbox_tensor = torch.tensor([
                (tx1+tx2)/2./S, (ty1+ty2)/2./S,
                (tx2-tx1)/S,    (ty2-ty1)/S,
            ], dtype=torch.float32)
        else:
            bbox_tensor = torch.tensor([0.5,0.5,1.0,1.0], dtype=torch.float32)

        mask_path = os.path.join(self.annotations_dir, "trimaps", name+".png")
        if os.path.exists(mask_path):
            mask_np = np.array(Image.open(mask_path)).astype(np.int64) - 1
            mask_np = np.clip(mask_np, 0, 2).astype(np.uint8)
        else:
            mask_np = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        mask_np     = np.array(Image.fromarray(mask_np).resize((self.img_size,self.img_size), Image.NEAREST))
        mask_tensor = torch.tensor(mask_np.copy(), dtype=torch.long)

        return {
            "image": image_tensor,
            "label": torch.tensor(class_id, dtype=torch.long),
            "bbox":  bbox_tensor,
            "mask":  mask_tensor,
            "name":  name,
        }

def get_dataloaders(root="./pets_data", img_size=224, batch_size=32,
                    num_workers=2, val_fraction=0.15, seed=42):
    tr = OxfordPetsDataset(root,"train",img_size,val_fraction,seed)
    va = OxfordPetsDataset(root,"val",  img_size,val_fraction,seed)
    te = OxfordPetsDataset(root,"test", img_size,val_fraction,seed)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True, drop_last=True),
        DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
        DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
    )
'''

with open('data/pets_dataset.py', 'w') as f:
    f.write(code.strip())
print('Written:', os.path.getsize('data/pets_dataset.py'), 'bytes')

## Cell 4 — Sanity check dataset

In [ ]:
import os, sys
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')

for k in list(sys.modules.keys()):
    if 'pets' in k or k == 'data': del sys.modules[k]

from data.pets_dataset import prepare_dataset, get_dataloaders

root = prepare_dataset('./pets_data')
train_loader, val_loader, test_loader = get_dataloaders(root=root, batch_size=8, num_workers=2)

b = next(iter(train_loader))
print('image :', b['image'].shape, b['image'].dtype)
print('label :', b['label'])
print('bbox  :', b['bbox'][0])
print('mask  :', b['mask'].shape, b['mask'].dtype, 'unique:', b['mask'].unique())
print()
print('DATASET OK - ready to train')

## Cell 5 — W&B login

In [ ]:
import wandb
wandb.login()

## Cell 6 — Mount Drive and restore checkpoints

In [ ]:
from google.colab import drive
import shutil, os
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/da6401_a2_checkpoints'
LOCAL = '/content/da6401_assignment_2/checkpoints'
os.makedirs(DRIVE, exist_ok=True)

restored = []
for f in os.listdir(DRIVE):
    if f.endswith('.pth'):
        shutil.copy(os.path.join(DRIVE,f), os.path.join(LOCAL,f))
        restored.append(f)

print('Restored from Drive:', restored or 'nothing yet')
print('Local checkpoints  :', os.listdir(LOCAL))

## Cell 7 — Define save_to_drive helper

In [ ]:
import shutil, os

def save_to_drive():
    DRIVE = '/content/drive/MyDrive/da6401_a2_checkpoints'
    LOCAL = '/content/da6401_assignment_2/checkpoints'
    saved = []
    for f in os.listdir(LOCAL):
        if f.endswith('.pth'):
            shutil.copy(os.path.join(LOCAL,f), os.path.join(DRIVE,f))
            saved.append(f)
    print('Saved to Drive:', saved)

print('save_to_drive() ready')

---
## Task 1 — Classification (3 runs for dropout ablation)
---

### Cell 8 — Task 1 Run 1: No dropout

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_cls.py --epochs 30 --lr 1e-4 --batch_size 32 --dropout_p 0.0

In [ ]:
save_to_drive()

### Cell 9 — Task 1 Run 2: Dropout p=0.2

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_cls.py --epochs 30 --lr 1e-4 --batch_size 32 --dropout_p 0.2

In [ ]:
save_to_drive()

### Cell 10 — Task 1 Run 3: Dropout p=0.5  ← saves best_cls.pth

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_cls.py --epochs 30 --lr 1e-4 --batch_size 32 --dropout_p 0.5

In [ ]:
save_to_drive()

---
## Task 2 — Localization
---

### Cell 11 — Task 2: Bounding box regression

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_loc.py --epochs 20 --lr 5e-4 --batch_size 32 --freeze_backbone False

In [ ]:
save_to_drive()

---
## Task 3 — Segmentation (3 transfer learning strategies)
---

### Cell 12 — Task 3 Strategy 1: Frozen backbone

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_seg.py --epochs 30 --lr 1e-3 --batch_size 16 --strategy frozen

In [ ]:
save_to_drive()

### Cell 13 — Task 3 Strategy 2: Partial fine-tuning

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_seg.py --epochs 30 --lr 1e-3 --batch_size 16 --strategy partial

In [ ]:
save_to_drive()

### Cell 14 — Task 3 Strategy 3: Full fine-tuning

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_seg.py --epochs 30 --lr 1e-3 --batch_size 16 --strategy full

In [ ]:
save_to_drive()

---
## Task 4 — Unified Multi-Task Pipeline
---

### Cell 15 — Task 4: Single forward pass — all three tasks

In [ ]:
import os
os.chdir('/content/da6401_assignment_2')
!python train_multi.py --epochs 30 --lr 5e-4 --batch_size 16 --load_cls True

In [ ]:
save_to_drive()

---
## W&B Visualizations for Report

Run these only after all training is done and checkpoints are restored.

---

### Cell 16 — Feature maps (section 2.4)

In [ ]:
import os, sys, torch, wandb
import matplotlib.pyplot as plt
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')
from data.pets_dataset import prepare_dataset, get_dataloaders
from models.vgg11 import VGG11

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root   = prepare_dataset('./pets_data')
_, val_loader, _ = get_dataloaders(root=root, batch_size=4, num_workers=2)

model = VGG11(num_classes=37).to(device)
model.load_state_dict(torch.load('./checkpoints/best_cls.pth', map_location=device)['model_state'])
model.eval()

img = next(iter(val_loader))['image'][0:1].to(device)
acts = {}
def hook(name):
    def fn(m,i,o): acts[name] = o.detach().cpu()
    return fn
h1 = model.block1.register_forward_hook(hook('block1'))
h5 = model.block5.register_forward_hook(hook('block5'))
with torch.no_grad(): model(img)
h1.remove(); h5.remove()

def show_maps(feat, title):
    maps = feat[0][:16]
    fig, axes = plt.subplots(4, 4, figsize=(12,12))
    for i, ax in enumerate(axes.flatten()):
        if i < len(maps): ax.imshow(maps[i].numpy(), cmap='viridis')
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    return fig

wandb.init(project='da6401-assignment2', name='feature_maps')
wandb.log({
    'feature_maps/block1_low_level': wandb.Image(show_maps(acts['block1'], 'Block 1 - edges and colors')),
    'feature_maps/block5_high_level': wandb.Image(show_maps(acts['block5'], 'Block 5 - semantic features')),
})
wandb.finish()
print('Feature maps logged')

### Cell 17 — BBox prediction table (section 2.5)

In [ ]:
import os, sys, torch, wandb, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')
from data.pets_dataset import prepare_dataset, get_dataloaders
from models.vgg11 import VGG11
from models.localization import LocalizationModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root   = prepare_dataset('./pets_data')
_, val_loader, _ = get_dataloaders(root=root, batch_size=16, num_workers=2)

vgg   = VGG11(num_classes=37)
model = LocalizationModel(vgg).to(device)
model.load_state_dict(torch.load('./checkpoints/best_loc.pth', map_location=device)['model_state'])
model.eval()

batch = next(iter(val_loader))
imgs  = batch['image'].to(device)
gt    = batch['bbox']
with torch.no_grad():
    pred = model(imgs).cpu()

def iou(p, t, eps=1e-6):
    px1,py1,px2,py2 = p[0]-p[2]/2,p[1]-p[3]/2,p[0]+p[2]/2,p[1]+p[3]/2
    tx1,ty1,tx2,ty2 = t[0]-t[2]/2,t[1]-t[3]/2,t[0]+t[2]/2,t[1]+t[3]/2
    inter = max(0,min(px2,tx2)-max(px1,tx1))*max(0,min(py2,ty2)-max(py1,ty1))
    union = (px2-px1)*(py2-py1)+(tx2-tx1)*(ty2-ty1)-inter
    return float(inter/(union+eps))

S=224
mean_t=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std_t =torch.tensor([0.229,0.224,0.225]).view(3,1,1)

wandb.init(project='da6401-assignment2', name='bbox_table')
table = wandb.Table(columns=['image','iou','pred_box','gt_box'])

for i in range(min(10, len(pred))):
    p_np = pred[i].numpy(); t_np = gt[i].numpy()
    iou_val = iou(p_np, t_np)
    img_np  = (imgs[i].cpu()*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()

    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(img_np)
    for box, color, lbl in [(t_np,'green','GT'),(p_np,'red','Pred')]:
        cx,cy,w,h = box
        ax.add_patch(patches.Rectangle(((cx-w/2)*S,(cy-h/2)*S),w*S,h*S,
                     linewidth=2,edgecolor=color,facecolor='none',label=lbl))
    ax.legend(fontsize=8); ax.set_title(f'IoU={iou_val:.3f}',fontsize=9); ax.axis('off')
    plt.tight_layout()
    table.add_data(wandb.Image(fig), round(iou_val,4),
                   str(np.round(p_np,3).tolist()), str(np.round(t_np,3).tolist()))
    plt.close(fig)

wandb.log({'detection/bbox_table': table})
wandb.finish()
print('BBox table logged')

### Cell 18 — Segmentation samples (section 2.6)

In [ ]:
import os, sys, torch, wandb, numpy as np
import matplotlib.pyplot as plt
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')
from data.pets_dataset import prepare_dataset, get_dataloaders
from models.vgg11 import VGG11
from models.segmentation import SegmentationModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root   = prepare_dataset('./pets_data')
_, val_loader, _ = get_dataloaders(root=root, batch_size=8, num_workers=2)

vgg   = VGG11(num_classes=37)
model = SegmentationModel(vgg, num_classes=3).to(device)
model.load_state_dict(torch.load('./checkpoints/best_seg_full.pth', map_location=device)['model_state'])
model.eval()

batch = next(iter(val_loader))
imgs  = batch['image'].to(device)
masks = batch['mask']
with torch.no_grad():
    preds = model(imgs).argmax(1).cpu()

COLORS = {0:(0.9,0.2,0.2),1:(0.2,0.6,0.9),2:(0.9,0.8,0.2)}
def to_rgb(m):
    out=np.zeros((*m.shape,3))
    for c,col in COLORS.items(): out[m==c]=col
    return out

mean_t=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std_t =torch.tensor([0.229,0.224,0.225]).view(3,1,1)

wandb.init(project='da6401-assignment2', name='seg_samples')
log_imgs=[]
for i in range(5):
    img_np=(imgs[i].cpu()*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    axes[0].imshow(img_np);                    axes[0].set_title('Original');     axes[0].axis('off')
    axes[1].imshow(to_rgb(masks[i].numpy()));  axes[1].set_title('Ground Truth'); axes[1].axis('off')
    axes[2].imshow(to_rgb(preds[i].numpy()));  axes[2].set_title('Predicted');    axes[2].axis('off')
    plt.tight_layout()
    log_imgs.append(wandb.Image(fig,caption=f'Sample {i+1}'))
    plt.close(fig)
wandb.log({'segmentation/samples':log_imgs})
wandb.finish()
print('Segmentation samples logged')

### Cell 19 — Wild pet inference (section 2.7)

In [ ]:
import os, sys, torch, wandb, numpy as np, requests
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from io import BytesIO
import albumentations as A
from albumentations.pytorch import ToTensorV2
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')
from models.multitask import MultiTaskModel

WILD = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/800px-YellowLabradorLooking_new.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/800px-Cat_November_2010-1a.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_Breeds.jpg/800px-Dog_Breeds.jpg',
]
BREEDS=['Abyssinian','Bengal','Birman','Bombay','British Shorthair','Egyptian Mau',
        'Maine Coon','Persian','Ragdoll','Russian Blue','Siamese','Sphynx',
        'American Bulldog','American Pit Bull Terrier','Basset Hound','Beagle',
        'Boxer','Chihuahua','English Cocker Spaniel','English Setter',
        'German Shorthaired','Great Pyrenees','Havanese','Japanese Chin',
        'Keeshond','Leonberger','Miniature Pinscher','Newfoundland','Pomeranian',
        'Pug','Saint Bernard','Samoyed','Scottish Terrier','Shiba Inu',
        'Staffordshire Bull Terrier','Wheaten Terrier','Yorkshire Terrier']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MultiTaskModel(num_classes=37,dropout_p=0.5,seg_classes=3).to(device)
model.load_state_dict(torch.load('./checkpoints/best_multitask.pth',map_location=device)['model_state'])
model.eval()

tfm = A.Compose([A.Resize(224,224),
                 A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)),
                 ToTensorV2()])
COLORS={0:(0.9,0.2,0.2),1:(0.2,0.6,0.9),2:(0.9,0.8,0.2)}
def to_rgb(m):
    out=np.zeros((*m.shape,3))
    for c,col in COLORS.items(): out[m==c]=col
    return out
mean_t=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std_t =torch.tensor([0.229,0.224,0.225]).view(3,1,1)

wandb.init(project='da6401-assignment2',name='wild_inference')
log_imgs=[]
for url in WILD:
    img_np=np.array(Image.open(BytesIO(requests.get(url,timeout=15).content)).convert('RGB'))
    inp=tfm(image=img_np)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        cls_log,bbox,seg_log=model(inp)
    breed=BREEDS[cls_log.argmax(1).item()]
    conf=torch.softmax(cls_log,1).max().item()
    p=bbox[0].cpu().numpy(); seg=seg_log.argmax(1)[0].cpu().numpy()
    vis=(inp[0].cpu()*std_t+mean_t).permute(1,2,0).clamp(0,1).numpy()
    cx,cy,w,h=p; S=224
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].imshow(vis)
    axes[0].add_patch(patches.Rectangle(((cx-w/2)*S,(cy-h/2)*S),w*S,h*S,
                      linewidth=2,edgecolor='red',facecolor='none'))
    axes[0].set_title(f'{breed}  {conf:.1%}',fontsize=10); axes[0].axis('off')
    axes[1].imshow(to_rgb(seg)); axes[1].set_title('Segmentation'); axes[1].axis('off')
    plt.tight_layout()
    log_imgs.append(wandb.Image(fig,caption=breed))
    plt.close(fig)
wandb.log({'wild/outputs':log_imgs})
wandb.finish()
print('Wild inference logged')